# Phase 10 — Interactive Plotly Visualization (60-class VoteNet, epoch-35 model)
Same workflow as Phase 9: auto-curate BEST / DENSEST / WORST / TINY-object
scenes, render each inline, then batch-export standalone HTMLs.

Color code:
 GREEN  = correct prediction (right class, IoU >= threshold with a GT)
 ORANGE = wrong class (right location)
 RED    = false positive
 BLACK dashed  = ground truth (found)
 YELLOW dashed = ground truth (MISSED)

Prereqs in checkpoints/: val_predictions-2.pkl (downloaded from the
40-epoch train+eval notebook output, phase10_eval/val_predictions.pkl).

In [ ]:
# Cell 0 — ONE-TIME: convert corner-tuple pkl -> Phase 9 viewer schema
import pickle
from pathlib import Path

import numpy as np

ROOT    = Path('/Users/dosvatsky/3D Object Detection')
RAW_PKL = ROOT / 'checkpoints/val_predictions-2.pkl'
VIS_PKL = ROOT / 'checkpoints/val_predictions_60class_v4.pkl'
VAL_DIR = ROOT / 'data/synthetic_v4_25d/val'
CLASSES = ROOT / 'data/synthetic_v4_25d/classes.txt'
PC_KEEP = 10000          # points stored per scene (viewer subsamples to 4000)

if VIS_PKL.exists():
    print('viewer pkl already exists — skipping conversion')
else:
    assert RAW_PKL.exists(), f'put the downloaded pkl at {RAW_PKL}'
    class_names = [l.split('\t')[-1].strip() for l in open(CLASSES) if l.strip()]
    with open(RAW_PKL, 'rb') as f:
        D = pickle.load(f)
    ids = sorted(f.name.replace('_pc.npz', '') for f in VAL_DIR.glob('*_pc.npz')
                 if not f.name.startswith('._'))
    assert len(ids) == len(D['preds']), \
        f'{len(ids)} scenes vs {len(D["preds"])} prediction lists'

    def eval_to_viewer(c):
        """SUN eval frame -> viewer frame (Z-up): (x, -y, z)."""
        c = np.asarray(c, np.float32)
        return np.stack([c[:, 0], -c[:, 1], c[:, 2]], 1)

    rng = np.random.default_rng(0)
    scenes = []
    for i, sid in enumerate(ids):
        pc = np.load(VAL_DIR / f'{sid}_pc.npz')['pc'][:, :3].astype(np.float32)
        keep = rng.choice(len(pc), min(PC_KEEP, len(pc)), replace=False)
        scenes.append({
            'scan_idx': i,
            'point_cloud': pc[keep][:, [0, 2, 1]],   # Y-up storage -> Z-up viewer
            'predictions': [{'class_id': int(c), 'score': float(s),
                             'box': eval_to_viewer(cor)}
                            for c, cor, s in D['preds'][i]],
            'groundtruths': [{'class_id': int(c), 'box': eval_to_viewer(cor)}
                             for c, cor in D['gts'][i]],
        })
        if (i + 1) % 300 == 0:
            print(f'  {i + 1}/{len(ids)}')
    with open(VIS_PKL, 'wb') as f:
        pickle.dump({'class_names': class_names, 'scenes': scenes}, f)
    print(f'wrote {VIS_PKL.name} ({VIS_PKL.stat().st_size / 1e6:.0f} MB)')

In [ ]:
# Cell 1 — imports and paths
import sys
from pathlib import Path
import numpy as np

sys.path.insert(0, '/Users/dosvatsky/3D Object Detection/scripts')
from interactive_viz_plotly_v2 import show_scene, load_data, aabb_iou, to_corners

PKL_PATH = '/Users/dosvatsky/3D Object Detection/checkpoints/val_predictions_60class_v4.pkl'
OUT_DIR  = Path('/Users/dosvatsky/3D Object Detection/phase10_viz_output')
OUT_DIR.mkdir(exist_ok=True)

assert Path(PKL_PATH).exists(), f'pkl not found at {PKL_PATH} — run Cell 0 first'
print(f'pkl found: {Path(PKL_PATH).stat().st_size / 1e6:.1f} MB')

In [ ]:
# Cell 2 — load data and class metadata
data = load_data(PKL_PATH)
class_names = data['class_names']
print(f'Total scenes: {len(data["scenes"])}')
print(f'Classes:      {len(class_names)}')

CLASS_REAL_SIZE = {
    'bed':2.0,'table':1.5,'sofa':2.0,'chair':0.55,'toilet':0.6,'desk':1.4,
    'dresser':1.0,'night_stand':0.55,'bookshelf':0.85,'bathtub':1.6,
    'ammo_box':0.35,'binoculars':0.22,'combat_knife':0.30,'flashlight':0.18,
    'gas_mask':0.28,'hand_grenade':0.12,'helmet':0.28,'magazine':0.18,
    'military_radio':0.30,'pistol':0.22,'rifle':0.95,'rocket_launcher':1.20,
    'shotgun':0.95,'sniper_rifle':1.20,'tactical_backpack':0.55,
    'tactical_vest':0.50,'wire_cutter':0.25,'axe':0.60,'barbed_wire_coil':0.90,
    'baton':0.55,'canteen':0.20,'claymore_mine':0.22,'concrete_barrier':2.00,
    'crossbow':0.75,'duffel_bag':0.80,'entrenching_shovel':0.60,
    'field_telephone':0.30,'first_aid_kit':0.30,'flare_gun':0.25,
    'fuel_drum':0.90,'grenade_launcher':0.75,'hedgehog':1.40,'jerry_can':0.47,
    'machete':0.65,'machine_gun':1.25,'military_boots':0.32,'military_cot':1.90,
    'military_drone':0.90,'military_shield':1.30,'mortar_tube':1.30,
    'night_vision_goggles':0.20,'propane_tank':0.60,'rifle_case':1.20,
    'sandbag':0.65,'smoke_grenade':0.15,'stretcher':2.10,'submachine_gun':0.60,
    'tank_mine':0.33,'tank_shell':0.90,'weapon_rack':1.80,
}

FURNITURE_IDS = set(range(10))   # class indices 0-9 are furniture
SMALL_CLASS_IDS = {i for i, c in enumerate(class_names)
                   if CLASS_REAL_SIZE.get(c, 999) < 0.35}
print(f'small class IDs (< 35cm): {sorted(SMALL_CLASS_IDS)}')

In [ ]:
# Cell 3 — auto-compute per-scene statistics (~30 sec)
def scene_stats(scene, score_thresh=0.5, match_iou=0.25):
    preds = sorted([p for p in scene['predictions'] if p['score'] >= score_thresh],
                   key=lambda p: -p['score'])
    gts = scene['groundtruths']

    pred_status = [None] * len(preds)
    gt_status   = ['missed'] * len(gts)
    pairs = []
    for i, p in enumerate(preds):
        pc = to_corners(p['box'])
        for j, g in enumerate(gts):
            gc = to_corners(g['box'])
            iou = aabb_iou(pc, gc)
            if iou >= match_iou:
                pairs.append((iou, i, j))
    pairs.sort(key=lambda t: -t[0])
    used_p, used_g = set(), set()
    for iou, i, j in pairs:
        if i in used_p or j in used_g:
            continue
        used_p.add(i); used_g.add(j)
        if preds[i]['class_id'] == gts[j]['class_id']:
            pred_status[i] = 'correct';  gt_status[j] = 'found'
        else:
            pred_status[i] = 'wrong';    gt_status[j] = 'missed'
    for i, st in enumerate(pred_status):
        if st is None:
            pred_status[i] = 'fp'

    n_gts = len(gts)
    n_correct = sum(1 for s in pred_status if s == 'correct')
    n_fp      = sum(1 for s in pred_status if s == 'fp')
    n_missed  = sum(1 for s in gt_status if s == 'missed')
    unique_gt_classes = len(set(g['class_id'] for g in gts))
    n_small = sum(1 for g in gts if g['class_id'] in SMALL_CLASS_IDS)
    is_furniture_only = n_gts > 0 and all(g['class_id'] in FURNITURE_IDS for g in gts)
    military_ratio = (sum(1 for g in gts if g['class_id'] not in FURNITURE_IDS)
                      / max(n_gts, 1))
    return {
        'scan_idx': scene['scan_idx'],
        'n_gts': n_gts,
        'n_preds': len(preds),
        'n_correct': n_correct,
        'n_fp': n_fp,
        'n_missed': n_missed,
        'precision': n_correct / max(len(preds), 1),
        'recall':    n_correct / max(n_gts, 1),
        'unique_classes': unique_gt_classes,
        'is_furniture_only': is_furniture_only,
        'military_ratio': military_ratio,
        'n_small': n_small,
    }

print(f'computing stats for {len(data["scenes"])} scenes...')
all_stats = [scene_stats(s) for s in data['scenes']]
print('done')

In [ ]:
# Cell 4 — curate categorical picks
def pick(stats, key, reverse=True, where=None):
    filt = [s for s in stats if (where(s) if where else True)]
    if not filt:
        return None
    return sorted(filt, key=lambda s: s[key], reverse=reverse)[0]

BEST      = pick(all_stats, 'n_correct',      True,  where=lambda s: s['n_gts'] >= 3)
DENSEST   = pick(all_stats, 'n_gts',          True)
DIVERSE   = pick(all_stats, 'unique_classes', True)
WORST     = pick(all_stats, 'n_missed',       True,  where=lambda s: s['n_gts'] >= 3)
FURNITURE = pick(all_stats, 'n_correct',      True,  where=lambda s: s['is_furniture_only'])
MILITARY  = pick(all_stats, 'military_ratio', True,
                 where=lambda s: s['n_gts'] >= 4 and s['military_ratio'] >= 0.75)
TINY      = pick(all_stats, 'n_small',        True,  where=lambda s: s['n_small'] >= 2)

picks = [
    ('BEST performance',        BEST),
    ('DENSEST scene',           DENSEST),
    ('MOST DIVERSE classes',    DIVERSE),
    ('WORST recall (honest)',   WORST),
    ('FURNITURE-only',          FURNITURE),
    ('MILITARY-heavy',          MILITARY),
    ('TINY-object case',        TINY),
]

for name, s in picks:
    if s is None:
        print(f'{name:22s} -> no scene matched criteria')
        continue
    print(f'{name:22s} -> Scene {s["scan_idx"]:4d}  '
          f'(gt={s["n_gts"]:2d}, correct={s["n_correct"]:2d}, '
          f'missed={s["n_missed"]:2d}, classes={s["unique_classes"]:2d}, '
          f'small={s["n_small"]:2d})')

In [ ]:
# Cell 5 — render each categorical pick inline
for name, s in picks:
    if s is None:
        continue
    print(f'\n=== Scene {s["scan_idx"]} — {name} ===')
    fig = show_scene(s['scan_idx'], score_threshold=0.5, top_k=20,
                     match_iou=0.25, pkl_path=PKL_PATH)
    fig.show()

In [ ]:
# Cell 6 — threshold experiments on the reference scene
REFERENCE_IDX = BEST['scan_idx']

for thresh in [0.3, 0.5, 0.7]:
    label = {'0.3': 'RELAXED (score >= 0.3)',
             '0.5': 'STANDARD (score >= 0.5)',
             '0.7': 'STRICT (score >= 0.7)'}[f'{thresh}']
    print(f'\n=== Scene {REFERENCE_IDX} — {label} ===')
    fig = show_scene(REFERENCE_IDX, score_threshold=thresh, top_k=30,
                     match_iou=0.25, pkl_path=PKL_PATH)
    fig.show()

In [ ]:
# Cell 7 — batch-export all curated scenes as standalone HTMLs
curated = []
for name, s in picks:
    if s is None:
        continue
    tag = name.lower().split()[0].replace('-', '_')
    curated.append((tag, s['scan_idx'], 0.5, name))

curated += [
    ('relaxed', REFERENCE_IDX, 0.3, f'Scene {REFERENCE_IDX} @ relaxed threshold'),
    ('strict',  REFERENCE_IDX, 0.7, f'Scene {REFERENCE_IDX} @ strict threshold'),
]

for tag, idx, thresh, name in curated:
    fig = show_scene(idx, score_threshold=thresh, top_k=30,
                     match_iou=0.25, pkl_path=PKL_PATH)
    out = OUT_DIR / f'phase10_{tag}_scene{idx:04d}_thr{int(thresh*100):02d}.html'
    fig.write_html(str(out))
    print(f'saved: {out.name}   ({name})')

print(f'\nAll {len(curated)} HTMLs in: {OUT_DIR}')

In [ ]:
# Cell 8 — free-form: render any specific scene (0-1499)
SCENE_IDX = 42
SCORE_THRESHOLD = 0.50
TOP_K = 20

fig = show_scene(SCENE_IDX, score_threshold=SCORE_THRESHOLD, top_k=TOP_K,
                 match_iou=0.25, pkl_path=PKL_PATH)
fig.show()

In [ ]:
# Cell 9 — find scenes containing a specific class (demo hunting)
def find_scenes_with_class(name, n=10):
    cls_idx = class_names.index(name)
    matches = []
    for sc in data['scenes']:
        for gt in sc['groundtruths']:
            if gt['class_id'] == cls_idx:
                matches.append(sc['scan_idx'])
                break
    return matches[:n]

for cls in ['pistol', 'hand_grenade', 'flashlight', 'helmet',
            'jerry_can', 'concrete_barrier', 'rifle', 'fuel_drum']:
    scenes = find_scenes_with_class(cls, n=5)
    print(f'{cls:22s} scenes: {scenes}')

In [ ]:
# Cell 10 — top-down (bird's-eye) view of any scene
SCENE_IDX = 42

fig = show_scene(SCENE_IDX, score_threshold=0.5, top_k=20,
                 match_iou=0.25, pkl_path=PKL_PATH,
                 view_preset='topdown')
fig.show()